# Data Load & Setting
### Setting

In [122]:
# 필수 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import catboost as cb

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


# 시각화 설정
plt.style.use('default')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'Apple SD Gothic Neo'  # 한글 폰트 설정 (MacOS 기준)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

# pandas 출력 옵션 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.colheader_justify', 'center')  # 컬럼 헤더 중앙 정렬

### Data Load

In [123]:
# 데이터 경로 설정
DATA_PATH = '../data/hotel_bookings.csv'
# COLUMNS_INFO_PATH = '../data/columns_info.csv'

In [124]:
# 데이터 파일 확인
print("◼︎ data 폴더 파일 확인")

file_path = '../data'
file_size = os.path.getsize(file_path)

for file in os.listdir(file_path):
    print(f"- {file}: {file_size} bytes")

◼︎ data 폴더 파일 확인
- hotel_bookings.csv: 128 bytes
- columns_info.csv: 128 bytes


In [125]:
df = pd.read_csv(DATA_PATH)
# df_columns = pd.read_csv(COLUMNS_INFO_PATH)

# Data Pre-processing

### N/A Processing

In [126]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [127]:
# inplace=True 대신 재할당 방식 사용 (pandas 3.0 호환)
df['children'] = df['children'].fillna(0)
df['children'] = df['children'].astype(dtype='int64')

df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               0
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

### New Feature

In [128]:
df['people'] = df['adults'] + df['children'] + df['babies']

In [129]:
def categorize_leadtime(days):
    """
    Lead Time을 구간별로 분류하는 함수
    days: 예약일과 도착일 사이의 일수
    반환값: 구간별 문자열

    == 구간 정의 ==
    0-7일 (1주 이내)
    8-30일 (1개월 이내)
    31-60일 (2개월 이내)
    61-90일 (3개월 이내)
    91-180일 (6개월 이내)
    180일 이상 (6개월 초과)
    """
    if days <= 7:
        return '0-7일 (1주 이내)'
    elif days <= 30:
        return '8-30일 (1개월 이내)'
    elif days <= 60:
        return '31-60일 (2개월 이내)'
    elif days <= 90:
        return '61-90일 (3개월 이내)'
    elif days <= 180:
        return '91-180일 (6개월 이내)'
    else:
        return '180일 이상 (6개월 초과)'
df['lead_time_category'] = df['lead_time'].apply(categorize_leadtime)
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               0
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [130]:
df['reservation_status_year'] = pd.to_datetime(df['reservation_status_date']).dt.year
df['reservation_status_month'] = pd.to_datetime(df['reservation_status_date']).dt.month
df['reservation_status_day'] = pd.to_datetime(df['reservation_status_date']).dt.day
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               0
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

### drop colums

In [131]:
# 사용 여부가 '미사용'인 컬럼명만 출력
not_using_columns = ['arrival_date_year',
                     'arrival_date_week_number',
                     'meal',
                     'country',
                     'market_segment',
                     'distribution_channel',
                     'is_repeated_guest',
                     'previous_cancellations',
                     'previous_bookings_not_canceled',
                     'assigned_room_type',
                     'agent',
                     'company',
                     'reservation_status',
                     'reservation_status_date',
                     'adults',
                     'children',
                     'babies']

df = df.drop(columns=not_using_columns)


### drop row

In [132]:
df = df[df['people'] != 0].reset_index(drop=True)
df = df[df['adr'] < 1000].reset_index(drop=True)
df

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,reserved_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,people,lead_time_category,reservation_status_year,reservation_status_month,reservation_status_day
0,Resort Hotel,0,342,July,1,0,0,C,3,No Deposit,0,Transient,0.00,0,0,2,180일 이상 (6개월 초과),2015,7,1
1,Resort Hotel,0,737,July,1,0,0,C,4,No Deposit,0,Transient,0.00,0,0,2,180일 이상 (6개월 초과),2015,7,1
2,Resort Hotel,0,7,July,1,0,1,A,0,No Deposit,0,Transient,75.00,0,0,1,0-7일 (1주 이내),2015,7,2
3,Resort Hotel,0,13,July,1,0,1,A,0,No Deposit,0,Transient,75.00,0,0,1,8-30일 (1개월 이내),2015,7,2
4,Resort Hotel,0,14,July,1,0,2,A,0,No Deposit,0,Transient,98.00,0,1,2,8-30일 (1개월 이내),2015,7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119204,City Hotel,0,23,August,30,2,5,A,0,No Deposit,0,Transient,96.14,0,0,2,8-30일 (1개월 이내),2017,9,6
119205,City Hotel,0,102,August,31,2,5,E,0,No Deposit,0,Transient,225.43,0,2,3,91-180일 (6개월 이내),2017,9,7
119206,City Hotel,0,34,August,31,2,5,D,0,No Deposit,0,Transient,157.71,0,4,2,31-60일 (2개월 이내),2017,9,7
119207,City Hotel,0,109,August,31,2,5,A,0,No Deposit,0,Transient,104.40,0,0,2,91-180일 (6개월 이내),2017,9,7


In [133]:
df['is_canceled'] = df['is_canceled'].astype('category')
df

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,reserved_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,people,lead_time_category,reservation_status_year,reservation_status_month,reservation_status_day
0,Resort Hotel,0,342,July,1,0,0,C,3,No Deposit,0,Transient,0.00,0,0,2,180일 이상 (6개월 초과),2015,7,1
1,Resort Hotel,0,737,July,1,0,0,C,4,No Deposit,0,Transient,0.00,0,0,2,180일 이상 (6개월 초과),2015,7,1
2,Resort Hotel,0,7,July,1,0,1,A,0,No Deposit,0,Transient,75.00,0,0,1,0-7일 (1주 이내),2015,7,2
3,Resort Hotel,0,13,July,1,0,1,A,0,No Deposit,0,Transient,75.00,0,0,1,8-30일 (1개월 이내),2015,7,2
4,Resort Hotel,0,14,July,1,0,2,A,0,No Deposit,0,Transient,98.00,0,1,2,8-30일 (1개월 이내),2015,7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119204,City Hotel,0,23,August,30,2,5,A,0,No Deposit,0,Transient,96.14,0,0,2,8-30일 (1개월 이내),2017,9,6
119205,City Hotel,0,102,August,31,2,5,E,0,No Deposit,0,Transient,225.43,0,2,3,91-180일 (6개월 이내),2017,9,7
119206,City Hotel,0,34,August,31,2,5,D,0,No Deposit,0,Transient,157.71,0,4,2,31-60일 (2개월 이내),2017,9,7
119207,City Hotel,0,109,August,31,2,5,A,0,No Deposit,0,Transient,104.40,0,0,2,91-180일 (6개월 이내),2017,9,7


### One-hot incoding

In [140]:
# 범주형 컬럼 추출 (is_canceled 제외)
from sklearn.preprocessing import LabelEncoder

cat_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_columns = [col for col in cat_columns if col != 'is_canceled']

# 라벨 인코딩
label_encoders = {}
for col in cat_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # 나중에 역변환이 필요할 경우를 위해 저장

print(f"라벨 인코딩된 컬럼 ({len(cat_columns)}개): {cat_columns}")
df

라벨 인코딩된 컬럼 (0개): []


,is_canceled,lead_time,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,people,reservation_status_year,reservation_status_month,reservation_status_day,hotel_City Hotel,hotel_Resort Hotel,arrival_date_month_April,arrival_date_month_August,arrival_date_month_December,arrival_date_month_February,arrival_date_month_January,arrival_date_month_July,arrival_date_month_June,arrival_date_month_March,arrival_date_month_May,arrival_date_month_November,arrival_date_month_October,arrival_date_month_September,reserved_room_type_A,reserved_room_type_B,reserved_room_type_C,reserved_room_type_D,reserved_room_type_E,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,lead_time_category_0-7일 (1주 이내),lead_time_category_180일 이상 (6개월 초과),lead_time_category_31-60일 (2개월 이내),lead_time_category_61-90일 (3개월 이내),lead_time_category_8-30일 (1개월 이내),lead_time_category_91-180일 (6개월 이내)
0,0,0.464043,0.000000,0.000000,0.00,0.166667,0.0,0.012355,0.0,0.0,0.018519,0.333333,0.545455,0.000000,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False
1,0,1.000000,0.000000,0.000000,0.00,0.222222,0.0,0.012355,0.0,0.0,0.018519,0.333333,0.545455,0.000000,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False
2,0,0.009498,0.000000,0.000000,0.02,0.000000,0.0,0.157597,0.0,0.0,0.000000,0.333333,0.545455,0.033333,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False
3,0,0.017639,0.000000,0.000000,0.02,0.000000,0.0,0.157597,0.0,0.0,0.000000,0.333333,0.545455,0.033333,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
4,0,0.018996,0.000000,0.000000,0.04,0.000000,0.0,0.202138,0.0,0.2,0.018519,0.333333,0.545455,0.066667,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119204,0,0.031208,0.966667,0.105263,0.10,0.000000,0.0,0.198536,0.0,0.0,0.018519,1.000000,0.727273,0.166667,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
119205,0,0.138399,1.000000,0.105263,0.10,0.000000,0.0,0.448914,0.0,0.4,0.037037,1.000000,0.727273,0.200000,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True
119206,0,0.046133,1.000000,0.105263,0.10,0.000000,0.0,0.317770,0.0,0.8,0.018519,1.000000,0.727273,0.200000,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False
119207,0,0.147897,1.000000,0.105263,0.10,0.000000,0.0,0.214532,0.0,0.0,0.018519,1.000000,0.727273,0.2

In [136]:
# 수치형 컬럼 추출 (is_canceled는 타겟 변수이므로 제외)
num_columns = df.select_dtypes(include=[np.number]).columns.tolist()
num_columns

['lead_time',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'booking_changes',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'people',
 'reservation_status_year',
 'reservation_status_month',
 'reservation_status_day']

In [137]:
scaler = MinMaxScaler()
df[num_columns] = scaler.fit_transform(df[num_columns])
df

,is_canceled,lead_time,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,people,reservation_status_year,reservation_status_month,reservation_status_day,hotel_City Hotel,hotel_Resort Hotel,arrival_date_month_April,arrival_date_month_August,arrival_date_month_December,arrival_date_month_February,arrival_date_month_January,arrival_date_month_July,arrival_date_month_June,arrival_date_month_March,arrival_date_month_May,arrival_date_month_November,arrival_date_month_October,arrival_date_month_September,reserved_room_type_A,reserved_room_type_B,reserved_room_type_C,reserved_room_type_D,reserved_room_type_E,reserved_room_type_F,reserved_room_type_G,reserved_room_type_H,reserved_room_type_L,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,lead_time_category_0-7일 (1주 이내),lead_time_category_180일 이상 (6개월 초과),lead_time_category_31-60일 (2개월 이내),lead_time_category_61-90일 (3개월 이내),lead_time_category_8-30일 (1개월 이내),lead_time_category_91-180일 (6개월 이내)
0,0,0.464043,0.000000,0.000000,0.00,0.166667,0.0,0.012355,0.0,0.0,0.018519,0.333333,0.545455,0.000000,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False
1,0,1.000000,0.000000,0.000000,0.00,0.222222,0.0,0.012355,0.0,0.0,0.018519,0.333333,0.545455,0.000000,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False
2,0,0.009498,0.000000,0.000000,0.02,0.000000,0.0,0.157597,0.0,0.0,0.000000,0.333333,0.545455,0.033333,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False
3,0,0.017639,0.000000,0.000000,0.02,0.000000,0.0,0.157597,0.0,0.0,0.000000,0.333333,0.545455,0.033333,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
4,0,0.018996,0.000000,0.000000,0.04,0.000000,0.0,0.202138,0.0,0.2,0.018519,0.333333,0.545455,0.066667,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119204,0,0.031208,0.966667,0.105263,0.10,0.000000,0.0,0.198536,0.0,0.0,0.018519,1.000000,0.727273,0.166667,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
119205,0,0.138399,1.000000,0.105263,0.10,0.000000,0.0,0.448914,0.0,0.4,0.037037,1.000000,0.727273,0.200000,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True
119206,0,0.046133,1.000000,0.105263,0.10,0.000000,0.0,0.317770,0.0,0.8,0.018519,1.000000,0.727273,0.200000,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False
119207,0,0.147897,1.000000,0.105263,0.10,0.000000,0.0,0.214532,0.0,0.0,0.018519,1.000000,0.727273,0.2

In [138]:
def preprocess_data

SyntaxError: expected '(' (1080107476.py, line 1)

## Modeling

In [ ]:
# def train_model(df):
#     # 1. X, y 분리
#     target = 'is_canceled'
#     if target not in df.columns:
#         raise ValueError(f"Target column '{target}' not found in dataset.")
        
#     X = df.drop(target, axis=1)
#     y = df[target]
    
#     # 2. Train / Test Split (75:25)
#     X_train, X_test, y_train, y_test = train_test_split(
#         X, y, test_size=0.25, random_state=42, stratify=y
#     )
    
#     print(f"\nTraining set size: {X_train.shape}")
#     print(f"Test set size: {X_test.shape}")
    
#     # 3. CatBoost 모델 정의 (Baseline 하이퍼파라미터)
#     model = cb.CatBoostClassifier(
#         iterations=100,
#         learning_rate=0.1,
#         depth=6,
#         random_state=42,
#         loss_function='Logloss',
#         verbose=False  # 학습 로그 출력 비활성화
#     )
    
#     # 4. 학습
#     print("\nStarting training...")
#     model.fit(X_train, y_train)
#     print("Training completed.")
    
#     # 5. 예측 및 평가
#     y_pred = model.predict(X_test)
#     accuracy = accuracy_score(y_test, y_pred)
    
#     print("-" * 40)
#     print(f"Model Accuracy: {accuracy:.4f}")
#     print("-" * 40)
#     print("Classification Report:\n")
#     print(classification_report(y_test, y_pred))
    
#     return model

# if __name__ == "__main__":
#     # 데이터 로드
#     raw_data = df
    
#     if raw_data is not None:
#         # 전처리
#         processed_data = preprocess_data(raw_data)
        
#         # 학습 및 평가
#         model = train_model(processed_data)